# 8.6 - RAG with LlamaIndex

**Module 8 - RAG Systems** | Netsetos GenAI Engineering

This notebook rebuilds the RAG pipeline you hand-wrote in 8.1 and rebuilt with LangChain in 8.5 - this time in about five lines with LlamaIndex, the data-specialist framework. You start with `VectorStoreIndex.from_documents` and `as_query_engine`, then work through the data-layer toolkit LlamaIndex is known for: custom chunking, one-line chat memory, the Document-to-Node data model, LlamaParse for hostile PDFs, a caching IngestionPipeline, response-synthesis modes, decompose-and-route query engines, FunctionAgent orchestration, and the production LangChain hybrid. Everything targets the current stack: `GoogleGenAI` + `gemini-embedding-001` on LlamaIndex 0.14.x.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup - reproducibility marker

A single comment that flags the lesson uses seeded values so results are stable across runs. It does nothing on its own - it is a placeholder header for the setup that follows.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

This is a no-op comment cell, not executable logic. It exists to mark the top of the setup block; there is no random seed actually set here, just a note that the lesson's examples are meant to be deterministic.

**How the code works - step by step**
- **`# Reproducibility ...`** - a lone comment; running it produces nothing.

**In one line:** a header comment, safe to run and skip past.

**What you'll see:** (no output - it is a comment only)

## Setup - install dependencies

Installs every package the lesson touches: LlamaIndex core, the Google GenAI LLM and embedding integrations, a HuggingFace embedder for the multilingual demo, LlamaParse's cloud client, and async helpers. The line is commented out so it only runs when you uncomment it on Colab or a fresh environment.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai llama-index-embeddings-huggingface sentence-transformers llama-cloud-services python-dotenv nest_asyncio -q  # noqa


**Explanation**

Environment prep, not a model call. It is a commented `pip install` covering the full lesson so no later cell fails on a missing import; uncomment it once per fresh machine.

**How the code works - step by step**
- **`llama-index`** - the core framework (VectorStoreIndex, Document, Settings, node parsers).
- **`llama-index-llms-google-genai` / `-embeddings-google-genai`** - the `GoogleGenAI` LLM and `GoogleGenAIEmbedding` used throughout.
- **`llama-index-embeddings-huggingface` + `sentence-transformers`** - the BGE-M3 multilingual embedder in the production cell.
- **`llama-cloud-services`** - the `LlamaParse` client for the PDF-parsing cell.
- **`python-dotenv`** - loads keys from a `.env` file.
- **`nest_asyncio`** - lets `asyncio.run()` work inside a notebook for the FunctionAgent cell.

**In one line:** one commented install line that covers every package the notebook needs.

**What you'll see:** (no output - environment prep)

## Setup - load your API key

Reads the Google AI Studio key from the environment (or a `.env` file) instead of hardcoding it. `setdefault` only fills the variable if it is empty, so an already-set key is left untouched.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY` from aistudio.google.com)

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

A credentials step, not a model call. It ensures `GOOGLE_API_KEY` exists in the process environment so every later `GoogleGenAI` / `GoogleGenAIEmbedding` call can authenticate; any one provider key is enough to start.

**How the code works - step by step**
- **`import os`** - access to environment variables.
- **`os.environ.setdefault("GOOGLE_API_KEY", "")`** - keeps an existing key, otherwise leaves an empty placeholder for you to fill.

**In one line:** make sure the Google key is in the environment before any Gemini call.

**What you'll see:** (no output - it only sets an environment variable; leaving it blank will make later Gemini calls fail until you supply a real key)

## 1 - RAG in 5 lines with VectorStoreIndex

This is the whole framework on one screen: one call chunks, embeds, and stores your documents, and a second call wires retrieval and synthesis into a query engine. It is the same pipeline you wrote in ~60 lines (8.1) and ~15 lines (8.5), now about five - the fastest way to see what LlamaIndex hides.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`)

In [ ]:
# LLAMAINDEX RAG - the 8.1/8.5 pipeline in ~5 lines
# pip install llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

# Configure once, globally. Settings replaced the removed ServiceContext.
Settings.llm = GoogleGenAI(model="gemini-2.5-flash")
Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001")

docs = [
    Document(text="Netsetos refund policy: full refund within 7 days, 50% from 8-30 days, none after 30."),
    Document(text="GenAI course: 14,999 rupees. Lifetime access, Discord, weekly live sessions, certificate."),
    Document(text="Live classes daily 7 PM IST on YouTube. Recordings in 2 hours. Python + GCP."),
    Document(text="Prerequisites: basic Python and high-school math. No ML experience needed."),
]

index = VectorStoreIndex.from_documents(docs)      # ONE call: chunk + embed + store
engine = index.as_query_engine(similarity_top_k=3)  # retrieve + synthesize

for q in ["What is the refund policy?", "How much does the course cost?"]:
    resp = engine.query(q)
    print(f"Q: {q}\nA: {resp}\n")

# Output:
# Q: What is the refund policy?
# A: Full refund within 7 days, 50% from 8-30 days, and no refund after 30 days.
# Q: How much does the course cost?
# A: The GenAI course costs 14,999 rupees, with lifetime access included.

**Explanation**

The express build. `Settings` configures the LLM and embedder once globally (it replaced the removed `ServiceContext`); `VectorStoreIndex.from_documents` does chunk+embed+store in a single call, and `as_query_engine` does retrieve+synthesize in another. Read it as: documents in, answers out, with every intermediate step handled for you.

**How the code works - step by step**
- **`Settings.llm` / `Settings.embed_model`** - set the generator (`gemini-2.5-flash`) and embedder (`gemini-embedding-001`) once for the whole session.
- **`docs = [Document(...)]`** - four short Netsetos policy/pricing strings as raw sources.
- **`VectorStoreIndex.from_documents(docs)`** - the single confident cut: chunks each doc into nodes, embeds them, stores the vectors.
- **`index.as_query_engine(similarity_top_k=3)`** - builds the retrieve-then-synthesize engine, pulling the top 3 nodes per question.
- **`engine.query(q)`** - runs the loop for two questions and prints each answer.

**In one line:** configure once -> `from_documents` to index -> `as_query_engine` to answer.

**What you'll see:** Two Q/A pairs print: the refund question returns the tiered policy (full within 7 days, 50% days 8-30, none after 30) and the cost question returns 14,999 rupees with lifetime access.

## 2 - Customize the pipeline: chunking, prompt, top-k

The default cut is not always right for your documents, so this cell re-grinds three knobs: a `SentenceSplitter` for chunk size and overlap, a grounded `PromptTemplate` that forces answers from context only, and `similarity_top_k` for how many chunks feed the answer. Chunk size is the single highest-leverage knob - it decides what one unit of retrieval even is.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`)

In [ ]:
# LLAMAINDEX - a customized pipeline
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.prompts import PromptTemplate
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

Settings.llm = GoogleGenAI(model="gemini-2.5-flash")
Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001")

# Small chunks = precise match, less context each; overlap saves split sentences
splitter = SentenceSplitter(chunk_size=128, chunk_overlap=20)

docs = [
    Document(text="Netsetos is an edtech company in Hyderabad offering GenAI, Data Science, and Cloud courses.", metadata={"source": "about.txt"}),
    Document(text="Refund policy: full refund within 7 days, 50% from 8-30 days, none after 30.", metadata={"source": "refund.txt"}),
    Document(text="The GenAI course costs 14,999 rupees with lifetime access and 5,000 GCP credits.", metadata={"source": "pricing.txt"}),
]

index = VectorStoreIndex.from_documents(docs, transformations=[splitter])

qa_prompt = PromptTemplate(
    "You are a Netsetos support assistant. Answer ONLY from the context.\n"
    "If it is not there, say you do not have that information.\n\n"
    "Context:\n{context_str}\n\nQuestion: {query_str}\nAnswer:"
)
engine = index.as_query_engine(similarity_top_k=2, text_qa_template=qa_prompt)

for q in ["What courses does Netsetos offer?", "Do you offer blockchain courses?"]:
    resp = engine.query(q)
    sources = [n.metadata.get("source", "?") for n in resp.source_nodes]
    print(f"Q: {q}\nA: {resp}\nsources: {sources}\n")

# Output:
# Q: What courses does Netsetos offer?  A: GenAI, Data Science, and Cloud.  sources: ['about.txt']
# Q: Do you offer blockchain courses?   A: I do not have that information.  sources: ['about.txt', 'pricing.txt']

**Explanation**

The same five-line skeleton with control handed back. A `SentenceSplitter` is passed as a transformation to change how documents are cut, a custom prompt grounds the model against hallucination, and per-source metadata is tracked so you can see which document each answer came from.

**How the code works - step by step**
- **`SentenceSplitter(chunk_size=128, chunk_overlap=20)`** - small precise chunks, with a 20-token overlap so a sentence split across a boundary is not lost.
- **`Document(..., metadata={"source": ...})`** - each doc carries a source label that flows to its nodes.
- **`from_documents(docs, transformations=[splitter])`** - builds the index using the custom splitter instead of the default.
- **`PromptTemplate(...)`** - a grounded prompt with `{context_str}` and `{query_str}` slots that tells the model to answer only from context.
- **`as_query_engine(similarity_top_k=2, text_qa_template=qa_prompt)`** - retrieves 2 chunks and synthesizes with the grounded prompt.
- **`n.metadata.get("source")`** - reads back which source files answered each question.

**In one line:** turn the chunk, prompt, and top-k knobs so the pipeline fits your query and refuses to guess.

**What you'll see:** The courses question answers "GenAI, Data Science, and Cloud" sourced from about.txt; the blockchain question - absent from the docs - returns "I do not have that information" instead of a hallucinated yes, proving the grounding prompt works.

## 3 - Conversational RAG in one line

A plain query engine forgets every question, but conversation needs history so "how much does *it* cost?" knows what "it" is. LlamaIndex folds memory, query-condensing, and retrieval into a single parameter: `as_chat_engine(chat_mode="condense_plus_context")`. In 8.5 this took a MessagesPlaceholder and a history list you managed by hand.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`)

In [ ]:
# LLAMAINDEX CHAT ENGINE - conversational RAG in one line
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

Settings.llm = GoogleGenAI(model="gemini-2.5-flash")
Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001")

docs = [
    Document(text="Refund policy: full refund within 7 days, 50% from 8-30 days, none after 30."),
    Document(text="GenAI course: 14,999 rupees, 146 hours, 14 modules, Python + GCP."),
    Document(text="Prerequisites: basic Python and high-school math. No ML experience needed."),
]
index = VectorStoreIndex.from_documents(docs)

# One line: memory + condensing + retrieval
chat = index.as_chat_engine(chat_mode="condense_plus_context")

for q in ["What are the prerequisites?", "And how much does it cost?", "Can I get a refund after 2 weeks?"]:
    resp = chat.chat(q)
    print(f"You: {q}\nBot: {resp}\n")

# Output:
# You: What are the prerequisites?        Bot: Basic Python and high-school math; no ML needed.
# You: And how much does it cost?         Bot: 14,999 rupees.  ("it" resolved to the course)
# You: Can I get a refund after 2 weeks?  Bot: Yes - a 50% refund applies between days 8 and 30.

**Explanation**

The chat-engine feature in one argument. `condense_plus_context` condenses each follow-up plus the running history into one standalone question, retrieves for it, and answers - so pronouns like "it" resolve automatically and you never build a message list yourself.

**How the code works - step by step**
- **`VectorStoreIndex.from_documents(docs)`** - builds the index as usual over three policy/pricing/prereq docs.
- **`index.as_chat_engine(chat_mode="condense_plus_context")`** - the whole feature: memory + condensing + retrieval in one call.
- **`chat.chat(q)`** - called per turn; the engine carries prior turns forward so later questions can reference earlier ones.
- The loop runs three dependent turns: prerequisites, then "how much does *it* cost?", then a refund-timing follow-up.

**In one line:** one `as_chat_engine` argument replaces the manual history plumbing.

**What you'll see:** Three conversational turns print: prerequisites (basic Python + high-school math), then "14,999 rupees" with "it" correctly resolved to the course, then a 50% refund answer for the two-week question - each turn using context from the ones before.

## 4 - The data model: Document -> Node -> Index

Everything so far rests on one abstraction, the Node. This cell splits a single `Document` into `Node`s by hand so you can inspect what a node actually is - text plus inherited metadata plus relationships to its source and neighbours - then builds the index straight from those nodes. Understanding the Node is what makes the "5 lines" legible.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`, for the embedder)

In [ ]:
# THE DATA MODEL - Document -> Node -> Index -> QueryEngine
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001")

doc = Document(
    text="Netsetos GenAI course: 14,999 rupees for lifetime access, a Discord community, and 5,000 GCP credits. The course spans 14 modules over 146 hours, covering Python and math foundations, RAG systems, fine-tuning, tool use, agents, and LLMOps in production. Refund policy: full refund within 7 days, 50 percent from day 8 to day 30, and no refund after 30 days.",
    metadata={"source": "faq.txt"},
)

# A Document is split into Nodes: chunk + metadata + relationships
nodes = SentenceSplitter(chunk_size=64, chunk_overlap=10).get_nodes_from_documents([doc])
n = nodes[0]
print("node text :", n.text[:48])
print("metadata  :", n.metadata)               # inherited from the Document
print("relations :", list(n.relationships.keys()))  # SOURCE, NEXT, PREVIOUS...

index = VectorStoreIndex(nodes)                # build straight from nodes
print("chunks indexed:", len(nodes))

# Output:
# node text : Netsetos GenAI course: 14,999 rupees for lifetime
# metadata  : {'source': 'faq.txt'}
# relations : [<NodeRelationship.SOURCE>, <NodeRelationship.NEXT>]  (repr abbreviated)
# chunks indexed: 2

**Explanation**

An inspection cell that opens the black box. Instead of letting `from_documents` make nodes invisibly, it calls the splitter directly, prints one node's internals, and builds `VectorStoreIndex` from the node list - showing the atom that retrieval and synthesis actually operate on.

**How the code works - step by step**
- **`Document(text=..., metadata={"source": "faq.txt"})`** - one longer source with a source tag.
- **`SentenceSplitter(chunk_size=64, chunk_overlap=10).get_nodes_from_documents([doc])`** - splits the document into Nodes explicitly (small size forces more than one).
- **`n.text` / `n.metadata` / `n.relationships`** - inspect the first node: its chunk text, the metadata it inherited from the Document, and its links (SOURCE, NEXT, PREVIOUS).
- **`VectorStoreIndex(nodes)`** - builds the index directly from the node list - the same atoms `from_documents` would have made.

**In one line:** a Document becomes Nodes (chunk + metadata + relationships), and the index is built from those Nodes.

**What you'll see:** Prints the first node's truncated text ("Netsetos GenAI course: 14,999 rupees for lifetime"), its inherited metadata `{'source': 'faq.txt'}`, its relationship keys (SOURCE, NEXT), and `chunks indexed: 2` - the 64-token document split into two nodes.

## 5 - LlamaParse for hostile PDFs

Real documents - scanned invoices, multi-column contracts, tables nested in tables - break naive text extractors. LlamaParse uses vision-language models to read a page the way a human would, and you dial quality-vs-cost with a tier: Fast (1 credit/page), Cost Effective (3), Agentic (10, the default), Agentic Plus (45). It plugs into the standard reader as a file extractor.

> **Needs a LlamaCloud API key** (set `LLAMA_CLOUD_API_KEY` from cloud.llamaindex.ai) and a real PDF file.

In [ ]:
# LLAMAPARSE - GenAI-native parsing for the hardest PDFs
# pip install llama-cloud-services   (get a key at cloud.llamaindex.ai)
import os
from llama_cloud_services import LlamaParse
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

# Tier by cost/quality: Fast(1) / Cost Effective(3) / Agentic(10, default) / Agentic Plus(45) credits/page
parser = LlamaParse(
    api_key=os.environ["LLAMA_CLOUD_API_KEY"],
    result_type="markdown",               # Fast tier cannot do markdown - use Cost Effective+
    parse_mode="parse_page_with_agent",    # the Agentic tier: tables, charts, scanned pages
)

# Plug LlamaParse into the standard reader as a file_extractor
reader = SimpleDirectoryReader(
    input_files=["annual_report.pdf"],
    file_extractor={".pdf": parser},
)
docs = reader.load_data()
index = VectorStoreIndex.from_documents(docs)
print(f"parsed {len(docs)} document(s) with the Agentic tier (10 credits/page)")

# Output:
# parsed 1 document(s) with the Agentic tier (10 credits/page)
# (a vision-language model read the tables and charts into clean markdown)

**Explanation**

LlamaIndex's biggest data-layer differentiator, wired into the normal ingestion path. A `LlamaParse` parser is configured with a tier and output type, handed to `SimpleDirectoryReader` as the extractor for `.pdf`, and its parsed output flows into `VectorStoreIndex` exactly like any other document.

**How the code works - step by step**
- **`LlamaParse(api_key=..., result_type="markdown", parse_mode="parse_page_with_agent")`** - creates the parser at the Agentic tier, emitting markdown (note: the Fast tier cannot produce markdown).
- **`SimpleDirectoryReader(input_files=[...], file_extractor={".pdf": parser})`** - routes any `.pdf` through LlamaParse instead of a plain text extractor.
- **`reader.load_data()`** - runs the vision-language parse, returning clean documents with tables and charts read in.
- **`VectorStoreIndex.from_documents(docs)`** - indexes the parsed output like any other source.

**In one line:** point a tier-configured LlamaParse at your PDFs through the standard reader, then index normally.

**What you'll see:** Prints "parsed 1 document(s) with the Agentic tier (10 credits/page)" - the file is read by a vision-language model into clean markdown. (Requires a LlamaCloud key and an actual annual_report.pdf on disk to run.)

## 6 - IngestionPipeline with hash caching

Re-indexing is where naive RAG bleeds money: rebuild nightly over 10,000 docs and you pay for 10,000 embeddings even when 50 changed. The `IngestionPipeline` is a declarative list of transformations that caches per node+transformation hash, so a re-run recomputes only the documents whose content changed - turning nightly re-indexing from O(N) into O(changed).

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`, for the embedder)

In [ ]:
# INGESTIONPIPELINE - declarative processing with hash caching
from llama_index.core import Document
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

docs = [
    Document(text="Refund policy: full refund within 7 days.", doc_id="refund"),
    Document(text="The GenAI course costs 14,999 rupees.", doc_id="pricing"),
    Document(text="Live classes run daily at 7 PM IST.", doc_id="schedule"),
]

pipeline = IngestionPipeline(transformations=[
    SentenceSplitter(chunk_size=128, chunk_overlap=10),
    GoogleGenAIEmbedding(model_name="gemini-embedding-001"),
])

nodes = pipeline.run(documents=docs)      # first run: all 3 processed + embedded
pipeline.persist("./pipeline_cache")      # save the node+transform hash cache
print("first run  - processed:", len(nodes))

# Later: only one document changed
docs[1].text = "The GenAI course costs 14,999 rupees, lifetime access included."
pipeline.load("./pipeline_cache")
nodes = pipeline.run(documents=docs)      # the 2 unchanged docs are cache hits
print("second run - only the changed doc was reprocessed (2 cache hits)")

# Output:
# first run  - processed: 3
# second run - the 2 unchanged docs hit the cache (embeddings reused, not recomputed)

**Explanation**

A declarative, cache-aware replacement for the manual load-split-embed-store loop. You define the transformations once, run and persist the cache, then change one document and re-run - the unchanged documents are cache hits and skip re-embedding entirely.

**How the code works - step by step**
- **`Document(text=..., doc_id=...)`** - three docs with stable ids so the cache can match them across runs.
- **`IngestionPipeline(transformations=[SentenceSplitter(...), GoogleGenAIEmbedding(...)])`** - a declarative splitter-then-embed pipeline.
- **`pipeline.run(documents=docs)`** - first run processes and embeds all three.
- **`pipeline.persist("./pipeline_cache")`** - saves the node+transform hash cache to disk.
- **`docs[1].text = ...`** then **`pipeline.load(...)`** + **`pipeline.run(...)`** - after editing one doc, reload the cache and re-run: the two unchanged docs hit the cache and only the edited one is reprocessed.

**In one line:** hash every node+transform, so a re-run only re-embeds what actually changed.

**What you'll see:** Prints "first run - processed: 3" then "second run - only the changed doc was reprocessed (2 cache hits)" - the two untouched documents reuse their cached embeddings instead of recomputing them.

## 7 - Response synthesis modes

Retrieval gets the right nodes; synthesis turns them into an answer - and how you synthesize is a real dial most people never touch. `compact` (default) stuffs nodes into 1-2 calls, `refine` walks them one at a time (N calls), and `tree_summarize` merges them pairwise up a tree (log N calls). Same retrieval, very different latency, cost, and quality.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`)

In [ ]:
# RESPONSE SYNTHESIS MODES - same retrieval, different synthesizer
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

Settings.llm = GoogleGenAI(model="gemini-2.5-flash")
Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001")

docs = [Document(text=t) for t in [
    "Module 1 covers Python and math foundations.",
    "Module 8 covers RAG systems end to end.",
    "Module 14 covers LLMOps and evaluation.",
]]
index = VectorStoreIndex.from_documents(docs)

# Same nodes, three synthesizers - different call patterns
for mode in ["compact", "refine", "tree_summarize"]:
    engine = index.as_query_engine(response_mode=mode, similarity_top_k=3)
    resp = engine.query("Summarize what the course covers.")
    print(f"[{mode:15s}] {str(resp)[:64]}")

# Output:
# [compact        ] The course covers Python foundations, RAG, and LLMOps.   (1-2 LLM calls)
# [refine         ] ... refines the answer chunk by chunk in sequence         (N LLM calls)
# [tree_summarize ] ... merges chunk summaries pairwise up a tree             (log N LLM calls)

**Explanation**

A comparison harness that holds retrieval fixed and varies only the synthesizer. It builds one index, then spins up three query engines that differ solely in `response_mode`, and runs the identical query through each so you can see the call-pattern trade rather than the answers changing because of different chunks.

**How the code works - step by step**
- **`VectorStoreIndex.from_documents(docs)`** - one index over three module-description docs.
- **`for mode in ["compact", "refine", "tree_summarize"]`** - iterate the three synthesis strategies.
- **`as_query_engine(response_mode=mode, similarity_top_k=3)`** - same top-k retrieval, different synthesizer per iteration.
- **`engine.query("Summarize what the course covers.")`** - runs the identical query so only the synthesis mode differs.

**In one line:** same retrieved nodes, three synthesizers, three call patterns (1-2 vs N vs log N).

**What you'll see:** Three labelled lines print - compact (1-2 calls) gives a one-shot summary, refine (N calls) walks chunk by chunk in sequence, tree_summarize (log N calls) merges chunk summaries pairwise - the same content synthesized three ways.

## 8 - SubQuestionQueryEngine for comparisons

A single query engine answers single questions, but "compare Company A's Q3 revenue with Company B's" is really two lookups plus a synthesis - and may span two separate indexes. `SubQuestionQueryEngine` decomposes the query automatically, routes each sub-question to the right `QueryEngineTool`, and synthesizes a comparative answer. Routing quality lives entirely in the tool descriptions you write.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`)

In [ ]:
# SUBQUESTIONQUERYENGINE - decompose a comparison into sub-questions
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.tools import QueryEngineTool
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

Settings.llm = GoogleGenAI(model="gemini-2.5-flash")
Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001")

# Two separate indexes, one per company
idx_a = VectorStoreIndex.from_documents([Document(text="Company A Q3 revenue was 120 crore, up 12%.")])
idx_b = VectorStoreIndex.from_documents([Document(text="Company B Q3 revenue was 95 crore, up 20%.")])

tools = [
    QueryEngineTool.from_defaults(idx_a.as_query_engine(),
        name="company_a", description="Q3 financials and revenue for Company A"),
    QueryEngineTool.from_defaults(idx_b.as_query_engine(),
        name="company_b", description="Q3 financials and revenue for Company B"),
]

engine = SubQuestionQueryEngine.from_defaults(query_engine_tools=tools)
print(engine.query("Compare the Q3 revenue of Company A and Company B."))

# Output:
# What the engine does internally (illustrative):
#   "What is Company A Q3 revenue?" -> company_a
#   "What is Company B Q3 revenue?" -> company_b
# Synthesized: A earned 120 crore (up 12%), B 95 crore (up 20%); A is higher, B grew faster.

**Explanation**

Multi-hop retrieval as a built-in, not hand-rolled. Two separate indexes are each wrapped as a named, described `QueryEngineTool`; the SubQuestion engine reads those descriptions to split a comparison into per-source sub-questions, runs each against its matching tool, and combines the results.

**How the code works - step by step**
- **`idx_a` / `idx_b`** - two separate one-document indexes, one per company.
- **`QueryEngineTool.from_defaults(..., name=..., description=...)`** - wrap each index as a tool with a specific description ("Q3 financials and revenue for Company A") that drives routing.
- **`SubQuestionQueryEngine.from_defaults(query_engine_tools=tools)`** - builds the decomposing engine over both tools.
- **`engine.query("Compare the Q3 revenue of Company A and Company B.")`** - one comparison is split into two sub-questions, each routed to its tool, then synthesized.

**In one line:** decompose the comparison, route each part to its own index, recombine into one answer.

**What you'll see:** Prints the synthesized comparison - Company A at 120 crore (up 12%) versus Company B at 95 crore (up 20%), noting A is higher but B grew faster - after internally asking one revenue sub-question per company.

## 9 - FunctionAgent: the LLM picks the tool

When the path must be chosen at runtime, you need an agent. `FunctionAgent` hands the LLM a set of tools - a query engine, a calculator - and lets it decide which to call using the model's native function-calling, which is more reliable than ReAct prompting for capable models. Because agents are async and may fire several calls per turn, this is the module's first `asyncio` code.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`). Uses `nest_asyncio` so `asyncio.run()` works inside a notebook.

In [ ]:
# FUNCTIONAGENT - the LLM decides which tool to call (async, function-calling)
import asyncio
import nest_asyncio
nest_asyncio.apply()  # let asyncio.run() work inside a notebook's running event loop
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.tools import QueryEngineTool, FunctionTool
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

Settings.llm = GoogleGenAI(model="gemini-2.5-flash")
Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001")

index = VectorStoreIndex.from_documents([Document(text="Refund policy: full refund within 7 days.")])
policy_tool = QueryEngineTool.from_defaults(
    index.as_query_engine(), name="policy_search",
    description="Search Netsetos policies, pricing, and schedules.")

def add_days(start_day: int, days: int) -> int:
    """Add a number of days to a start day."""
    return start_day + days

agent = FunctionAgent(
    tools=[policy_tool, FunctionTool.from_defaults(add_days)],
    llm=GoogleGenAI(model="gemini-2.5-flash"),
    system_prompt="Answer Netsetos questions using the tools.",
)

async def main():
    print(await agent.run("What is the refund policy?"))

asyncio.run(main())

# Output:
# The agent calls policy_search -> "Full refund within 7 days." -> answers:
# You can get a full refund within 7 days of purchase.

**Explanation**

Runtime tool selection with no hand-written router. A query engine and a plain Python function are each wrapped as tools and given to a `FunctionAgent`; the LLM reads their descriptions and chooses which to invoke. The async wrapper is required because an agent may make several model and tool calls per turn.

**How the code works - step by step**
- **`nest_asyncio.apply()`** - lets `asyncio.run()` run inside the notebook's live event loop.
- **`QueryEngineTool.from_defaults(index.as_query_engine(), name="policy_search", description=...)`** - wraps the RAG engine as a searchable tool.
- **`FunctionTool.from_defaults(add_days)`** - wraps a plain typed Python function (its docstring becomes the tool description) as a second tool.
- **`FunctionAgent(tools=[...], llm=..., system_prompt=...)`** - the agent that lets the LLM choose between the tools via native function-calling.
- **`async def main()` + `await agent.run(...)` + `asyncio.run(main())`** - the async pattern: wrap in `async def`, launch with `asyncio.run()`.

**In one line:** give the LLM described tools and let it pick, running the whole thing asynchronously.

**What you'll see:** The agent selects the `policy_search` tool for a policy question, retrieves "Full refund within 7 days," and prints a natural-language answer that you can get a full refund within 7 days of purchase - with no router written by you.

## 10 - Production: persist, evaluate, multilingual, hybrid

Shipping LlamaIndex needs four things a demo skips: real persistence to an external store, evaluation before deploy to catch hallucinations, multilingual reach, and - the recommended architecture - the LangChain hybrid where a LlamaIndex engine becomes a LangChain tool via one adapter call. This cell demonstrates all four in sequence.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`). Swapping to BGE-M3 downloads a HuggingFace model on first run.

In [ ]:
# PRODUCTION LLAMAINDEX - persist, evaluate, multilingual, the LangChain hybrid
from llama_index.core import (VectorStoreIndex, Document, StorageContext,
                             load_index_from_storage, Settings)
from llama_index.core.evaluation import FaithfulnessEvaluator
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

Settings.llm = GoogleGenAI(model="gemini-2.5-flash")
Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001")

# 1. Persist and reload (external vector stores auto-persist in real production)
index = VectorStoreIndex.from_documents([Document(text="Refund within 7 days.")])
index.storage_context.persist("./storage")
index = load_index_from_storage(StorageContext.from_defaults(persist_dir="./storage"))

# 2. Evaluate for hallucination before shipping (full harness is Lesson 8.11)
resp = index.as_query_engine().query("What is the refund window?")
print("faithful:", FaithfulnessEvaluator().evaluate_response(response=resp).passing)

# 3. Multilingual: swap the embedder to BGE-M3 (100+ languages, incl. Hindi)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-m3")  # set BEFORE building a fresh multilingual index

# 4. The hybrid: expose a LlamaIndex engine as a LangChain tool for a LangGraph agent
# from llama_index.core.tools import QueryEngineTool
# lc_tool = QueryEngineTool.from_defaults(index.as_query_engine()).as_langchain_tool()
print("faithful check + BGE-M3 multilingual + LangChain hybrid ready")

# Output:
# faithful: True
# faithful check + BGE-M3 multilingual + LangChain hybrid ready

**Explanation**

The production checklist in one cell. It persists an index to disk and reloads it, runs a `FaithfulnessEvaluator` to check an answer is grounded, swaps the embedder to a multilingual BGE-M3 model, and points (in comments) to the `as_langchain_tool()` adapter that joins the two frameworks.

**How the code works - step by step**
- **`index.storage_context.persist("./storage")`** then **`load_index_from_storage(...)`** - save and reload the index (external vector stores auto-persist in real production).
- **`FaithfulnessEvaluator().evaluate_response(response=resp).passing`** - check the answer is faithful to its retrieved context before shipping (the full harness is Lesson 8.11).
- **`Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-m3")`** - swap to a 100+ language embedder, set before building a fresh multilingual index.
- **`# QueryEngineTool(...).as_langchain_tool()`** - the commented hybrid handle that exposes a LlamaIndex engine as a LangChain/LangGraph tool.

**In one line:** persist -> evaluate -> go multilingual -> compose with LangChain.

**What you'll see:** Prints "faithful: True" (the refund-window answer is grounded in its context) followed by "faithful check + BGE-M3 multilingual + LangChain hybrid ready" confirming all four production steps completed.

Across ten concepts you saw the same RAG pipeline shrink to five lines, then met the depth underneath it: the Node is the atom LlamaIndex stores and retrieves, chunk size and synthesis mode are the two highest-leverage dials, LlamaParse and the caching IngestionPipeline are the data-layer tools a general framework rarely matches, and SubQuestion/FunctionAgent handle multi-source and runtime-routed questions. The honest verdict is not "LlamaIndex beats LangChain" but "specialize, then compose" - LlamaIndex owns the data layer, LangChain owns orchestration, joined by `as_langchain_tool()`. Next: 8.7 extends routing to knowledge graphs (GraphRAG), 8.10 grows the FunctionAgent into a full agentic-RAG system, and 8.11 turns the FaithfulnessEvaluator into a golden-set CI harness.